# ModernBERT-large fine-tuning on GLUE/QNLI

Mục tiêu: fine-tune `answerdotai/ModernBERT-large` trên **GLUE/QNLI** và evaluate trên **validation/dev set** để đối chiếu với Table 5 của paper ModernBERT.

Theo Table 6 cho **ModernBERT-large / QNLI**:

- Learning rate: `3e-5`
- Weight decay: `5e-6`
- Epochs: `2`

Target tham khảo ở Table 5: **ModernBERT-large QNLI dev accuracy ≈ 95.2**.

Paper không công bố batch size trong Table 6, nên batch size được đặt theo VRAM Kaggle.


In [1]:
# ============================================================
# 1. Install / upgrade libraries
# ============================================================
# Sau khi chạy cell này, nếu Kaggle yêu cầu restart session thì restart rồi chạy lại từ đầu.
!pip -q install -U "transformers>=4.48.0" "datasets>=3.0.0" "accelerate>=0.34.0" evaluate scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 90.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 98.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 94.4 MB/s eta 0:00:00


In [2]:
# ============================================================
# 2. Imports and config
# ============================================================
import os
import gc
import json
import random
import inspect
from pathlib import Path

import numpy as np
import torch
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

print("torch:", torch.__version__)
import transformers, datasets
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)

MODEL_NAME = "answerdotai/ModernBERT-large"
DATASET_NAME = "nyu-mll/glue"
TASK_NAME = "qnli"
OUTPUT_DIR = "./modernbert-large-qnli"

# Paper Table 6: ModernBERT-large / QNLI
LR = 3e-5
WEIGHT_DECAY = 5e-6
NUM_EPOCHS = 2

# Batch size không được paper ghi rõ. Chỉnh theo VRAM Kaggle.
# Với ModernBERT-large trên Kaggle T4:
# - Nếu OOM: giảm train batch xuống 4 hoặc 2.
# - Nếu còn dư VRAM: có thể tăng lên 16.
PER_DEVICE_TRAIN_BATCH_SIZE = 16
PER_DEVICE_EVAL_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 2  # effective train batch size = 16

# QNLI thường không cần quá dài. 256 giúp ModernBERT-large chạy ổn hơn trên T4.
# Nếu GPU dư VRAM, có thể thử MAX_LENGTH = 512.
MAX_LENGTH = 256

SEED = 42

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_all_seeds(SEED)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("bf16 supported:", torch.cuda.is_bf16_supported())


torch: 2.10.0+cu128
transformers: 5.9.0
datasets: 4.8.5
CUDA available: True
GPU: Tesla T4
bf16 supported: True


In [3]:
# ============================================================
# 3. Load GLUE/QNLI dataset
# ============================================================
# QNLI fields: question, sentence, label
# Table 5 báo GLUE dev score, nên ta dùng validation/dev, không dùng test.

raw_datasets = load_dataset(DATASET_NAME, TASK_NAME)

print(raw_datasets)
print("Train example:", raw_datasets["train"][0])
print("Validation example:", raw_datasets["validation"][0])

label_names = raw_datasets["train"].features["label"].names
num_labels = len(label_names)
id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in enumerate(label_names)}

print("Labels:", label_names)
print("id2label:", id2label)


README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

qnli/train-00000-of-00001.parquet:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

qnli/validation-00000-of-00001.parquet:   0%|          | 0.00/872k [00:00<?, ?B/s]

qnli/test-00000-of-00001.parquet:   0%|          | 0.00/877k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/104743 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5463 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5463 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'sentence', 'label', 'idx'],
        num_rows: 104743
    })
    validation: Dataset({
        features: ['question', 'sentence', 'label', 'idx'],
        num_rows: 5463
    })
    test: Dataset({
        features: ['question', 'sentence', 'label', 'idx'],
        num_rows: 5463
    })
})
Train example: {'question': 'When did the third Digimon series begin?', 'sentence': 'Unlike the two seasons before it and most of the seasons that followed, Digimon Tamers takes a darker and more realistic approach to its story featuring Digimon who do not reincarnate after their deaths and more complex character development in the original Japanese.', 'label': 1, 'idx': 0}
Validation example: {'question': 'What came into force after the new constitution was herald?', 'sentence': 'As of that day, the new constitution heralding the Second Republic came into force.', 'label': 0, 'idx': 0}
Labels: ['entailment', 'not_entailment']
id2label:

In [4]:
# ============================================================
# 4. Tokenizer and preprocessing
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def preprocess_function(examples):
    return tokenizer(
        examples["question"],
        examples["sentence"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)

if "label" in tokenized_datasets["train"].column_names:
    tokenized_datasets = tokenized_datasets.rename_column("label", "labels")

keep_cols = {"input_ids", "attention_mask", "labels"}
if "token_type_ids" in tokenized_datasets["train"].column_names:
    keep_cols.add("token_type_ids")

remove_cols = [c for c in tokenized_datasets["train"].column_names if c not in keep_cols]
tokenized_datasets = tokenized_datasets.remove_columns(remove_cols)

print(tokenized_datasets)
print(tokenized_datasets["train"][0].keys())


config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

Map:   0%|          | 0/104743 [00:00<?, ? examples/s]

Map:   0%|          | 0/5463 [00:00<?, ? examples/s]

Map:   0%|          | 0/5463 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 104743
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 5463
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 5463
    })
})
dict_keys(['labels', 'input_ids', 'attention_mask'])


In [5]:
# ============================================================
# 5. Load ModernBERT-large for sequence classification
# ============================================================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

# Giảm VRAM cho ModernBERT-large trên Kaggle T4.
# Không thay đổi LR/WD/epochs, chỉ là kỹ thuật tiết kiệm bộ nhớ.
model.gradient_checkpointing_enable()

print("Loaded model:", MODEL_NAME)
print("num_labels:", model.config.num_labels)


model.safetensors:   0%|          | 0.00/1.58G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-large
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loaded model: answerdotai/ModernBERT-large
num_labels: 2


In [6]:
# ============================================================
# 6. Metric: GLUE/QNLI accuracy
# ============================================================

glue_metric = evaluate.load("glue", TASK_NAME)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return glue_metric.compute(predictions=predictions, references=labels)


In [7]:
# ============================================================
# 7. TrainingArguments
# ============================================================

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

print("use_bf16:", use_bf16)
print("use_fp16:", use_fp16)

ta_params = inspect.signature(TrainingArguments.__init__).parameters

args_dict = dict(
    output_dir=OUTPUT_DIR,

    # Hyperparameters theo Table 6: ModernBERT-large / QNLI
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    num_train_epochs=NUM_EPOCHS,

    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,

    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,

    seed=SEED,
    data_seed=SEED,

    fp16=use_fp16,
    bf16=use_bf16,

    report_to="none",
    save_total_limit=2,
)

if "eval_strategy" in ta_params:
    args_dict["eval_strategy"] = "epoch"
elif "evaluation_strategy" in ta_params:
    args_dict["evaluation_strategy"] = "epoch"

if "overwrite_output_dir" in ta_params:
    args_dict["overwrite_output_dir"] = True

# Table 6 không ghi warmup nên để 0 nếu version có tham số này.
if "warmup_ratio" in ta_params:
    args_dict["warmup_ratio"] = 0.0

training_args = TrainingArguments(**args_dict)
training_args


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


use_bf16: True
use_fp16: False


TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=True,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_us

In [8]:
# ============================================================
# 8. Fine-tune
# ============================================================

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

trainer_params = inspect.signature(Trainer.__init__).parameters
if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_params:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)

train_result = trainer.train()
print(train_result)


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy
1,0.732256,0.276825,0.949112
2,0.309284,0.337580,0.947831


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3274, training_loss=0.6262809523883649, metrics={'train_runtime': 11404.4853, 'train_samples_per_second': 18.369, 'train_steps_per_second': 0.287, 'total_flos': 4.46155801826884e+16, 'train_loss': 0.6262809523883649, 'epoch': 2.0})


In [9]:
# ============================================================
# 9. Evaluate on QNLI validation/dev set
# ============================================================

eval_results = trainer.evaluate(tokenized_datasets["validation"])
print(eval_results)

acc = eval_results["eval_accuracy"] * 100
print(f"QNLI validation/dev accuracy: {acc:.2f}")
print("Paper Table 5 ModernBERT-large QNLI target: about 95.2")

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
with open(Path(OUTPUT_DIR) / "eval_results_qnli_large.json", "w") as f:
    json.dump(eval_results, f, indent=2)


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Epoch,Accuracy
0.309284,0.276825,2,0.949112


{'eval_loss': 0.2768252193927765, 'eval_accuracy': 0.9491122094087497}
QNLI validation/dev accuracy: 94.91
Paper Table 5 ModernBERT-large QNLI target: about 95.2


In [10]:
# ============================================================
# 10. Save best model and tokenizer
# ============================================================

best_dir = "./modernbert-large-qnli-best"
trainer.save_model(best_dir)
tokenizer.save_pretrained(best_dir)

print("Best checkpoint saved to:", best_dir)
print("Trainer best model checkpoint:", trainer.state.best_model_checkpoint)
print("Best metric:", trainer.state.best_metric)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best checkpoint saved to: ./modernbert-large-qnli-best
Trainer best model checkpoint: ./modernbert-large-qnli/checkpoint-1637
Best metric: 0.9491122094087497


## Nếu bị OOM hoặc score chưa gần Table 5

1. Nếu bị CUDA OOM, giảm:
   ```python
   PER_DEVICE_TRAIN_BATCH_SIZE = 4
   PER_DEVICE_EVAL_BATCH_SIZE = 8
   GRADIENT_ACCUMULATION_STEPS = 4
   ```
   Effective batch vẫn xấp xỉ 16.

2. Nếu score thấp hơn rõ, thử đổi:
   ```python
   SEED = 123
   ```
   hoặc:
   ```python
   SEED = 3407
   ```

3. Nếu GPU dư VRAM, thử:
   ```python
   MAX_LENGTH = 512
   ```

4. ModernBERT-large target trong Table 5 là khoảng `95.2`, nên kết quả quanh `94.8 - 95.3` là có thể chấp nhận được khi reproduce trên Kaggle.
